<a href="https://colab.research.google.com/github/charlyacha/labo1-colabs/blob/main/03_Gaussiana_TCL_y_compatibilidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Colab 03 — La gaussiana, el Teorema Central del Límite y la compatibilidad entre mediciones**Laboratorio 1 · Clase 3****Objetivos.**1. Entender por qué la distribución de errores aleatorios tiende a ser normal.2. Superponer la gaussiana **predicha** por tus estimadores sobre tu histograma — y entender por qué   la predecimos en vez de ajustarla.3. Decidir cuantitativamente si dos mediciones de la misma magnitud son **compatibles**.**Requisitos previos:** Colabs 01 y 02.> **Antes de empezar:** hacé `Archivo → Guardar una copia en Drive`. Vas a trabajar sobre *tu* copia; el original queda intacto para el resto del curso.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltnp.random.seed(20260826)

---## 1. La función gaussiana$$ f(x) = \\frac{1}{\\sigma\\sqrt{2\\pi}} \\exp\\!\\left[-\\frac{(x-\\mu)^2}{2\\sigma^2}\\right] $$Tiene dos parámetros y ningún otro: el centro $\\mu$ y el ancho $\\sigma$. El prefactor está para queel área total valga 1, como corresponde a una densidad de probabilidad.

In [ ]:
def gaussiana(x, mu, sigma):    return np.exp(-(x - mu)**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi))x = np.linspace(-5, 5, 400)fig, ax = plt.subplots(figsize=(6.5, 4))for sg, est in zip([0.5, 1.0, 2.0], ['-', '--', ':']):    ax.plot(x, gaussiana(x, 0, sg), est, label=f'$\\sigma$ = {sg}')ax.set_xlabel('$x$'); ax.set_ylabel('$f(x)$')ax.set_title('Gaussianas centradas en $\\mu = 0$')ax.grid(alpha=0.3); ax.legend()fig.tight_layout(); plt.show()

In [ ]:
# ¿Cuánta probabilidad hay dentro de 1σ, 2σ, 3σ?from scipy.integrate import quadfor k in [1, 2, 3]:    p, _ = quad(gaussiana, -k, k, args=(0, 1))    print(f"±{k}σ  ->  {100*p:.2f} % de la probabilidad")

Esos números son la razón por la que un resultado a $3\\sigma$ de lo esperado llama la atención: si elmodelo fuera correcto, ocurriría en un 0,3 % de los casos.Ojo con el abuso de esta interpretación: vale **si** la distribución es efectivamente normal y **si**no hay errores sistemáticos. Las dos condiciones fallan seguido.

---## 2. Superponer la gaussiana predicha (no ajustada)Acá hay una decisión metodológica deliberada, y conviene que entiendas por qué.Podríamos *ajustar* una gaussiana al histograma y ver qué $\\mu$ y $\\sigma$ salen. Pero es másinformativo hacer lo contrario: tomar $\\bar{x}$ y $s$ —que ya calculaste en el Colab 02, sin ningunahipótesis sobre la forma de la distribución— y **predecir** con ellos la curva. Después mirás sidescribe los datos.Si describe bien: tus estimadores capturan la distribución, y la hipótesis de normalidad esrazonable. Si no describe bien: aprendiste algo real sobre tu proceso de medición. Ajustar la curvate habría dado siempre "la mejor gaussiana posible", incluso cuando ninguna gaussiana sirve.

In [ ]:
# --- DATOS DE EJEMPLO (reemplazar por los propios) ---t = np.random.normal(0.235, 0.032, 120)# ----------------------------------------------------N = len(t)xbar = np.mean(t)s = np.std(t, ddof=1)          # ddof=1, siempresem = s / np.sqrt(N)def bins_scott(x):    h = 3.49 * np.std(x, ddof=1) / len(x)**(1/3)    return max(1, int(np.ceil((x.max() - x.min()) / h)))fig, ax = plt.subplots(figsize=(7, 4.2))ax.hist(t, bins=bins_scott(t), density=True, edgecolor='k', alpha=0.65,        label='datos (normalizados)')xx = np.linspace(t.min() - 3*s, t.max() + 3*s, 400)ax.plot(xx, gaussiana(xx, xbar, s), 'crimson', lw=2,        label='gaussiana PREDICHA con $\\bar{x}$ y $s$')ax.set_xlabel('Tiempo de reacción [s]'); ax.set_ylabel('Densidad de probabilidad')ax.set_title('La curva no está ajustada: está predicha a partir de los estimadores')ax.grid(alpha=0.3); ax.legend()fig.tight_layout(); plt.show()

`density=True` en el histograma es imprescindible acá: normaliza el área a 1 para que las escalassean comparables. Sin eso, la curva teórica queda aplastada contra el eje.

---## 3. Teorema Central del Límite¿Por qué aparece la gaussiana en todos lados? Porque los errores experimentales rara vez tienen unasola causa: son la suma de muchas contribuciones pequeñas e independientes (fluctuaciones térmicas,vibración, ruido electrónico, tu propia mano). El **Teorema Central del Límite** dice que la suma demuchas variables aleatorias independientes, cualquiera sea su distribución individual, tiende a unagaussiana.Vamos a verlo. Partimos de una distribución **uniforme**, que no se parece en nada a una campana, ysumamos $k$ de ellas.

In [ ]:
M = 20000   # cantidad de repeticiones del experimentofig, axes = plt.subplots(1, 4, figsize=(14, 3.2))for ax, k in zip(axes, [1, 2, 5, 12]):    suma = np.random.uniform(-1, 1, size=(M, k)).sum(axis=1)    ax.hist(suma, bins=60, density=True, edgecolor='none', alpha=0.75)    # gaussiana predicha: var de U(-1,1) es 1/3, así que la suma de k tiene σ = sqrt(k/3)    xx = np.linspace(suma.min(), suma.max(), 300)    ax.plot(xx, gaussiana(xx, 0, np.sqrt(k/3)), 'crimson', lw=1.8)    ax.set_title(f'suma de k = {k}', fontsize=10)    ax.set_xlabel('valor'); ax.grid(alpha=0.3)axes[0].set_ylabel('densidad')fig.suptitle('Teorema Central del Límite: de uniforme a gaussiana', y=1.04)fig.tight_layout(); plt.show()

Con $k=1$ la distribución es un rectángulo. Con $k=2$ ya es un triángulo. Con $k=12$ esindistinguible de una gaussiana a ojo. Y nunca supusimos normalidad en ningún lado.> **Nota.** Acá usamos números aleatorios solo como recurso ilustrativo, dos celdas. No es una> simulación Monte Carlo como técnica de análisis de datos: eso es otra cosa y no forma parte de> este curso.

---## 4. ¿Dos mediciones son compatibles?Ésta es la pregunta que aparece cada vez que medís algo por dos métodos, o cuando comparás turesultado con el valor de tabla. La respuesta **no** es "sí, dan parecido".Definimos la **discrepancia normalizada**:$$ z = \\frac{|x_1 - x_2|}{\\sqrt{\\sigma_1^2 + \\sigma_2^2}} $$Es la diferencia medida en unidades de la incerteza combinada. Criterio operativo habitual:| $z$ | Lectura ||---|---|| $z < 1$ | compatibles; la diferencia es menor que el ruido || $1 \\le z < 2$ | compatibles dentro de lo esperable || $2 \\le z < 3$ | en tensión; conviene revisar || $z \\ge 3$ | incompatibles: o hay un sistemático, o alguna incerteza está subestimada |El caso más interesante es el último, y casi nunca significa que la física esté mal: casi siempresignifica que alguien subestimó su error.

In [ ]:
def compatibilidad(x1, dx1, x2, dx2, etiquetas=('medición 1', 'medición 2')):    z = abs(x1 - x2) / np.sqrt(dx1**2 + dx2**2)    if z < 1:      veredicto = "compatibles"    elif z < 2:    veredicto = "compatibles dentro de lo esperable"    elif z < 3:    veredicto = "en tensión — revisar"    else:          veredicto = "INCOMPATIBLES — sistemático o incerteza subestimada"    print(f"{etiquetas[0]}: {x1:.5g} ± {dx1:.2g}")    print(f"{etiquetas[1]}: {x2:.5g} ± {dx2:.2g}")    print(f"z = {z:.2f}  ->  {veredicto}")    return z# Ejemplo: g medida por dos métodoscompatibilidad(9.79, 0.04, 9.81, 0.03, ('caída libre', 'plano inclinado'))print()compatibilidad(9.62, 0.02, 9.81, 0.03, ('caída libre (con roce)', 'valor de referencia'))

El segundo caso da $z \\approx 5$. Ningún aumento de $N$ lo va a arreglar: es un error sistemático, yse corrige mejorando el experimento o modelando el efecto, no midiendo más veces.

---## 5. Ejercicios**3.1.** Con tus datos de tiempo de reacción, hacé el histograma normalizado con la gaussianapredicha. ¿La describe bien? Mirá especialmente las colas: ¿hay más datos lejos del centro de losque la gaussiana predice? (En tiempos de reacción es habitual una cola hacia valores altos —distracciones — que rompe la simetría.)**3.2.** Medí el período de un péndulo de dos maneras: (a) cronometrando una oscilación, 30 veces;(b) cronometrando 10 oscilaciones y dividiendo por 10, 3 veces. Calculá $T$ y su incerteza en cadacaso y usá `compatibilidad()`. ¿Cuál método da menor incerteza? ¿Por qué?**3.3.** Repetí el experimento del TCL partiendo de una distribución fuertemente asimétrica(`np.random.exponential`). ¿Cuántos términos hacen falta para que se vea gaussiana? ¿Más o menos quepartiendo de la uniforme?**3.4.** *(conceptual)* El TCL exige varianza finita. Buscá qué es la distribución de Cauchy y porqué el promedio de $N$ variables de Cauchy no mejora con $N$. Es el contraejemplo que muestra que laregla del $1/\\sqrt{N}$ no es magia: tiene hipótesis.